# EarthForge — TiTiler COG Visualization

This notebook shows how to combine EarthForge's STAC search and raster inspection
with a local [TiTiler](https://developmentseed.org/titiler/) instance to render
Cloud-Optimized GeoTIFFs as map tiles.

**Prerequisites:**
```bash
# Start TiTiler locally (from examples/web/):
cd examples/web && docker compose up -d
# TiTiler API docs: http://localhost:8000/docs
```

**Workflow:**
1. Search STAC for imagery and DEMs
2. Inspect COG metadata (dimensions, CRS, bands)
3. Build TiTiler tile URLs for visualization
4. Render inline map previews using `folium` (optional)
5. Generate shareable tile URLs for web maps

If you don't have Docker, the TiTiler URL construction still works —
you can point them at any public TiTiler instance.

In [ ]:
import sys
import json

sys.path.insert(0, "../../packages/core/src")
sys.path.insert(0, "../../packages/raster/src")
sys.path.insert(0, "../../packages/stac/src")

from earthforge.core.config import EarthForgeProfile
from earthforge.raster.info import inspect_raster
from earthforge.stac.search import search_catalog

profile = EarthForgeProfile(
    name="earth-search",
    stac_api="https://earth-search.aws.element84.com/v1",
    storage_backend="local",
)

# Local TiTiler instance
TITILER_URL = "http://localhost:8000"

print("EarthForge + TiTiler ready")

## 1. Discover COG Assets via STAC

Search for Sentinel-2 imagery and extract the COG asset URLs.
These URLs can be passed directly to TiTiler for on-the-fly rendering.

In [ ]:
# Search for clear Sentinel-2 scenes over Lexington, KY
results = await search_catalog(
    profile,
    collections=["sentinel-2-l2a"],
    bbox=[-84.6, 37.95, -84.4, 38.1],
    datetime_range="2025-07-01/2025-08-31",
    max_items=5,
)

# Pick the clearest scene
items = sorted(
    results.items,
    key=lambda i: i.properties.get("eo:cloud_cover", 100),
)
best = items[0]
cc = best.properties.get("eo:cloud_cover", "?")
print(f"Best scene: {best.id} (cloud: {cc}%)")
print(f"Date:       {best.datetime}")
print(f"Assets:     {[a.key for a in best.assets]}")

In [ ]:
# Extract band URLs
band_urls = {}
for asset in best.assets:
    if asset.media_type and "tiff" in asset.media_type.lower():
        band_urls[asset.key] = asset.href

print("Available COG bands:")
for key, url in sorted(band_urls.items()):
    print(f"  {key:<10} {url[:80]}...")

## 2. Inspect COG Before Visualization

Check the raster properties to choose the right rendering parameters
(band indices, rescale range, colormap).

In [ ]:
# Inspect the red band (B04)
red_url = band_urls.get("red") or band_urls.get("B04", "")
if red_url:
    info = await inspect_raster(red_url)
    print(f"Red band (B04):")
    print(f"  Dimensions: {info.width} x {info.height}")
    print(f"  CRS:        {info.crs}")
    print(f"  Bands:      {info.band_count}")
    print(f"  Dtype:      {info.bands[0].dtype if info.bands else '?'}")
    print(f"  Tiled:      {info.is_tiled}")
    print(f"  Compression: {info.compression}")
    print(f"  Overviews:  {info.overview_count}")

## 3. Build TiTiler Tile URLs

TiTiler serves COGs as map tiles on the fly. The key parameters:
- `url` — COG URL (must be HTTP-accessible from TiTiler)
- `bidx` — band indices (1-indexed): `1` for single band, `1,2,3` for RGB
- `rescale` — min,max for normalization (Sentinel-2 surface reflectance: 0-3000)
- `colormap_name` — matplotlib colormap for single-band visualization

In [ ]:
from urllib.parse import urlencode, quote


def titiler_tile_url(cog_url, bidx="1", rescale="0,3000", colormap=None):
    """Build a TiTiler XYZ tile URL template."""
    params = {"url": cog_url, "bidx": bidx, "rescale": rescale}
    if colormap:
        params["colormap_name"] = colormap
    qs = urlencode(params)
    return f"{TITILER_URL}/cog/tiles/WebMercatorQuad/{{z}}/{{x}}/{{y}}.png?{qs}"


def titiler_preview_url(cog_url, bidx="1", rescale="0,3000", colormap=None, max_size=512):
    """Build a TiTiler preview (single image) URL."""
    params = {"url": cog_url, "bidx": bidx, "rescale": rescale, "max_size": max_size}
    if colormap:
        params["colormap_name"] = colormap
    qs = urlencode(params)
    return f"{TITILER_URL}/cog/preview.png?{qs}"


# Single-band visualization (Red)
if red_url:
    tile_url = titiler_tile_url(red_url, bidx="1", rescale="0,3000")
    preview_url = titiler_preview_url(red_url, bidx="1", rescale="0,3000")
    print("Red band tile URL:")
    print(f"  {tile_url[:120]}...")
    print()
    print("Red band preview URL:")
    print(f"  {preview_url}")

## 4. Common Visualization Recipes

These are ready-to-use TiTiler URL patterns for common Sentinel-2 visualizations.

In [ ]:
# Build visualization URLs for different band combinations
visualizations = {}

# True color (RGB = B04, B03, B02)
for key in ["red", "B04"]:
    if key in band_urls:
        visualizations["true_color"] = {
            "description": "True Color (Red, Green, Blue)",
            "tile_url": titiler_tile_url(band_urls[key], bidx="1", rescale="0,3000"),
            "note": "Single-band red; for true RGB, use TiTiler's mosaic endpoint with B04+B03+B02",
        }
        break

# NIR false color (vegetation emphasis)
for key in ["nir", "B08"]:
    if key in band_urls:
        visualizations["nir"] = {
            "description": "Near-Infrared (vegetation reflection)",
            "tile_url": titiler_tile_url(band_urls[key], bidx="1", rescale="0,5000"),
        }
        break

# SWIR (moisture/burn detection)
for key in ["swir16", "B11"]:
    if key in band_urls:
        visualizations["swir"] = {
            "description": "Short-Wave Infrared (moisture/burn)",
            "tile_url": titiler_tile_url(band_urls[key], bidx="1", rescale="0,4000"),
        }
        break

# Scene Classification (SCL) — categorical
for key in ["scl", "SCL"]:
    if key in band_urls:
        visualizations["scene_class"] = {
            "description": "Scene Classification Layer",
            "tile_url": titiler_tile_url(band_urls[key], bidx="1", rescale="0,11"),
        }
        break

print("Visualization recipes:")
for name, viz in visualizations.items():
    print(f"\n  {name}: {viz['description']}")
    print(f"  Tile URL: {viz['tile_url'][:100]}...")

## 5. DEM Visualization with Terrain Colormap

Copernicus DEM tiles are single-band GeoTIFFs with elevation values.
TiTiler can render these with a `terrain` colormap for topographic visualization.

In [ ]:
# Search for Copernicus DEM over Kentucky
dem_results = await search_catalog(
    profile,
    collections=["cop-dem-glo-30"],
    bbox=[-85.5, 37.0, -84.0, 38.5],
    max_items=5,
)

print(f"DEM tiles found: {len(dem_results.items)}")
for item in dem_results.items:
    print(f"  {item.id}")
    for asset in item.assets:
        if asset.key in ("data", "dem"):
            # Build terrain visualization URL
            dem_tile_url = titiler_tile_url(
                asset.href, bidx="1", rescale="100,600", colormap="terrain"
            )
            print(f"    Terrain tile URL: {dem_tile_url[:100]}...")

## 6. Render Inline Map (requires folium)

If `folium` is installed, render an interactive map directly in the notebook.
The map loads tiles from TiTiler on the fly.

```bash
pip install folium
```

In [ ]:
try:
    import folium

    m = folium.Map(location=[38.05, -84.5], zoom_start=11)

    # Add Sentinel-2 Red band layer
    if "true_color" in visualizations:
        folium.TileLayer(
            tiles=visualizations["true_color"]["tile_url"],
            attr="EarthForge + TiTiler",
            name="Sentinel-2 Red",
            overlay=True,
            opacity=0.7,
        ).add_to(m)

    if "nir" in visualizations:
        folium.TileLayer(
            tiles=visualizations["nir"]["tile_url"],
            attr="EarthForge + TiTiler",
            name="Sentinel-2 NIR",
            overlay=True,
            opacity=0.7,
            show=False,
        ).add_to(m)

    folium.LayerControl().add_to(m)

    # Display map (works in Jupyter)
    display(m)
    print("Map rendered. Toggle layers in the top-right control.")

except ImportError:
    print("folium not installed. Install with: pip install folium")
    print("You can still use the tile URLs above in any web map library.")

## 7. Export Tile URLs as JSON

Save the visualization configurations as JSON for use in web applications
or the `titiler_map.html` viewer.

In [ ]:
from pathlib import Path

output_dir = Path("data/titiler")
output_dir.mkdir(parents=True, exist_ok=True)

config = {
    "titiler_url": TITILER_URL,
    "scene": {
        "id": best.id,
        "datetime": best.datetime,
        "cloud_cover": best.properties.get("eo:cloud_cover"),
    },
    "visualizations": {
        name: {"description": viz["description"], "tile_url": viz["tile_url"]}
        for name, viz in visualizations.items()
    },
    "band_urls": band_urls,
}

config_path = output_dir / "visualization_config.json"
config_path.write_text(json.dumps(config, indent=2))

print(f"Saved visualization config to {config_path}")
print(f"File size: {config_path.stat().st_size:,} bytes")
print()
print("Use these tile URLs in:")
print("  - examples/web/titiler_map.html (paste COG URL)")
print("  - Leaflet: L.tileLayer(tile_url)")
print("  - MapLibre: map.addSource('cog', {type:'raster', tiles:[tile_url]})")